## Spike analysis for Neuropixels 2.0 - Open Ephys streams - multiprobe
###### NB not yet tested with NP 1.0 or with single probes
###### Mar 03, 2026

### Import packages 

In [1]:
# spikeinterface

import spikeinterface.full as si
from spikeinterface.preprocessing import bandpass_filter, common_reference, phase_shift
import spikeinterface.widgets as sw

from spikeinterface.sorters import installed_sorters
from spikeinterface.sorters import get_default_sorter_params, get_sorter_params_description

from spikeinterface.sorters import run_sorter as ss
from spikeinterface.sorters import Kilosort4Sorter as ks

from spikeinterface.curation import apply_sortingview_curation  # optional
from spikeinterface.postprocessing import compute_principal_components
import spikeinterface.qualitymetrics as sqm

# probe
from probeinterface.plotting import plot_probe
from probeinterface import write_probeinterface, read_probeinterface
from probeinterface import ProbeGroup, write_prb, read_prb

# kilosort
import kilosort
from kilosort import io
from kilosort import run_kilosort, DEFAULT_SETTINGS
from kilosort.io import load_probe

# curation
import spikeinterface.curation as sc
# pip install huggingface_hub ## if not already installed in your environment
import huggingface_hub # import the classifier repo

# view traces
import ephyviewer

# extras
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import re
import os
import json
from itertools import product
import datetime # Get the current time 
import psutil # forces to kill processes
from matplotlib.colors import LinearSegmentedColormap

# Custom LFP colormap: green – white – purple (used for LFP map visualisation)
GWPmap = LinearSegmentedColormap.from_list(
    'BlueWhiteYellow',
    [
        (0.0, "#A1D76A"),
        (0.5, '#ffffff'),
        (1.0, "#CE6AD7")
    ],
    N=256
)

# sanity check for versions and availability of sorters
print("SpikeInterface version:", si.__version__)
print("Kilosort version:", kilosort.__version__)
print("Huggingface Hub version:", huggingface_hub.__version__)
# check which sorters are installed and get kilosort4 default parameters and descriptions
print(installed_sorters())
# kilosort4 is available as a sorter in spikeinterface, but it is not installed by default. You need to install it separately and make sure it is in your system path for spikeinterface to find it. If you have it installed, you can check its version and get its default parameters and descriptions as follows:
# print(ss.get_sorter_version("kilosort4"))
params = get_default_sorter_params(sorter_name_or_class='kilosort4')
print("Parameters:\n", params)

desc = get_sorter_params_description(sorter_name_or_class='kilosort4')
print("Descriptions:\n", desc)

# check widget backend
# sw.set_default_plotter_backend(backend="ipywidgets")
sw.set_default_plotter_backend(backend="matplotlib")
# activate the widget backend for interactive plotting in Jupyter notebooks
# %matplotlib widget
print(sw.get_default_plotter_backend())

SpikeInterface version: 0.103.2
Kilosort version: 4.1.5
Huggingface Hub version: 1.4.1
['kilosort4', 'simple', 'spykingcircus2', 'tridesclous2']
Parameters:
 {'batch_size': 60000, 'nblocks': 1, 'Th_universal': 9, 'Th_learned': 8, 'nt': 61, 'shift': None, 'scale': None, 'batch_downsampling': 1, 'artifact_threshold': inf, 'nskip': 25, 'whitening_range': 32, 'highpass_cutoff': 300, 'binning_depth': 5, 'sig_interp': 20, 'drift_smoothing': [0.5, 0.5, 0.5], 'nt0min': None, 'dmin': None, 'dminx': 32, 'min_template_size': 10, 'template_sizes': 5, 'nearest_chans': 10, 'nearest_templates': 100, 'max_channel_distance': 32, 'max_peels': 100, 'templates_from_data': True, 'n_templates': 6, 'n_pcs': 6, 'Th_single_ch': 6, 'acg_threshold': 0.2, 'ccg_threshold': 0.25, 'cluster_neighbors': 10, 'cluster_downsampling': 20, 'max_cluster_subset': 25000, 'x_centers': None, 'cluster_init_seed': 5, 'duplicate_spike_ms': 0.25, 'position_limit': 100, 'do_CAR': True, 'invert_sign': False, 'save_extra_vars': False,

### Find dirs for the main loop

In [2]:
root_directory  = os.path.normpath(r"I:\Neuropixels_Backup\Agata")
dest_path       = os.path.normpath(r'X:\Agata_spikesorting') 
now             = datetime.datetime.now() # Format the time as YYYYMMDDHHMM 
formatted_time  = now.strftime('%Y%m%d%H%M')

# Initialize a dictionary to store the 1st and 2nd level directories
folders         = {1: []}

# Walk through the directory tree
for dirpath, dirnames, filenames in os.walk(root_directory):
    
    # Calculate the level (relative to root_directory)
    level = os.path.normpath(dirpath).replace(root_directory, "").strip(os.sep).count(os.sep) + 1

    # Store only the first two levels
    if level in folders:
        folders[level].extend([os.path.join(dirpath, d) for d in dirnames])

max_depth = max(len(path.split(os.sep)) for path in folders[1])
folders2 = [path for path in folders[1] if len(path.split(os.sep)) == max_depth]

# filter only ####-##-##_##-##-## folders using regexp
pattern = r"\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}"
folders2 = [path for path in folders2 if re.search(pattern, os.path.basename(path))]
# if any item in the list folder2 contains the substring "_test" in its name, remove it from the list
folders2 = [p for p in folders2 if "_test" not in str(p).lower()]
print(folders2)

['I:\\Neuropixels_Backup\\Agata\\20250311_Agata_VNS_N2_happy\\2025-03-11_14-25-44', 'I:\\Neuropixels_Backup\\Agata\\20250604_Agata_VNS_C3_stress\\2025-06-04_11-52-52', 'I:\\Neuropixels_Backup\\Agata\\20250520_Agata_VNS_C1_stress\\2025-05-20_12-52-33', 'I:\\Neuropixels_Backup\\Agata\\20250520_Agata_VNS_C2_stress\\2025-05-20_17-24-24', 'I:\\Neuropixels_Backup\\Agata\\20250606_Agata_VNS_C4_stress\\2025-06-06_12-14-46', 'I:\\Neuropixels_Backup\\Agata\\20250331_Agata_VNS_N4_happy\\2025-03-31_17-50-25', 'I:\\Neuropixels_Backup\\Agata\\20250331_Agata_VNS_N3_happy\\2025-03-31_12-55-04', 'I:\\Neuropixels_Backup\\Agata\\20250305_Agata_VNS_N1_happy\\2025-03-05_13-14-55', 'I:\\Neuropixels_Backup\\Agata\\20251219_Agata_VNS_N5_happy\\2025-12-19_12-15-05', 'I:\\Neuropixels_Backup\\Agata\\20251219_Agata_VNS_N6_happy\\2025-12-19_16-06-59']


In [ ]:
# # import spikeinterface.extractors as se

# folder_path = r"I:\Neuropixels_Backup\IDO1\20260219_ET1_rec1\2026-02-19_15-55-40"

# # List available streams
# streams = si.get_neo_streams('openephysbinary', folder_path)
# blocks = si.get_neo_num_blocks('openephysbinary', folder_path)
# print("Available blocks:", blocks)
# print("Available streams:", streams)
# # using regexp extract all the 'Probe' type (ProbeA, ProbeB) from streams (e.g. 'Record Node 101#Neuropix-PXI-100.ProbeA')
# stream_names, stream_ids = streams
# probes = [re.search(r'Probe\w+', n).group() for n in stream_names]
# print("Probes found:", probes)
# print("Available streams:", stream_ids)
# dest_path_rec           = os.path.join(dest_path, os.path.basename(folder_path), probes[0])
# print("Destination path for recording:", dest_path_rec)
# recName                 = os.path.basename(os.path.normpath(folder_path))
# mainDir                 = os.path.basename(os.path.dirname(folder_path))
# print("Recording name:", recName)
# print("Main directory:", mainDir)

### Main loop

In [ ]:
reRunKilosort   = False     # if True it will re-run kilosort even if the folder exists
reRunLFP        = False     # if True it will re-run LFP processing even if the folder exists
reRunAnalyzer   = False     # if True it will re-run sorting analyzer / phy export even if existing
reDoPreprocessing = False   # if True it will re-run raw preprocessing even if existing
min_duration    = 600       # minimum duration in seconds (10 minutes)
resampled_fs    = 1000      # target sampling frequency for LFP (after downsampling)
t_range_LFP     = (100, 110) # time range in seconds for LFP plotting (relative to the start of the recording)

# for dataPath in folders2:
dataPath = r"I:\Neuropixels_Backup\Agata\20251219_Agata_VNS_N6_happy\2025-12-19_16-06-59" # for testing
dataPath = os.path.normpath(dataPath)
print(f"\nProcessing: {dataPath}")


streams = si.get_neo_streams('openephysbinary', dataPath)
stream_names, stream_ids = streams

# Map streams per probe (NP1.0 has AP+LFP streams, NP2.0 typically only AP)
probe_streams = {}
for name, sid in zip(stream_names, stream_ids):
    m = re.search(r'Probe\w+', name)
    if not m:
        continue
    probe = m.group()
    entry = probe_streams.setdefault(probe, {'ap': None, 'lfp': None, 'other': []})
    lname = name.lower()
    if 'lfp' in lname:
        entry['lfp'] = sid
    elif 'ap' in lname or 'wideband' in lname:
        entry['ap'] = sid
    else:
        entry['other'].append(sid)

for probe, entry in probe_streams.items():
    if entry['ap'] is None and entry['other']:
        entry['ap'] = entry['other'][0]

probes = list(probe_streams.keys())
print("Probes found:", probes)
print("Streams by probe:", probe_streams)

# -------------------------------
# TTL Event extraction (once per recording, shared across probes)
# -------------------------------
parent_folder = os.path.basename(os.path.dirname(dataPath))
subfolder = os.path.basename(dataPath)
events_dest = os.path.join(dest_path, parent_folder, subfolder, 'events')
os.makedirs(events_dest, exist_ok=True)

events_OE = si.read_openephys_event(dataPath, block_index=0)
first_ap_id = next((v['ap'] for v in probe_streams.values() if v['ap'] is not None), stream_ids[0])
rec_t0    = si.read_openephys(dataPath, block_index=0, stream_id=first_ap_id)
t0        = rec_t0.get_times(segment_index=0)[0]
print(f"TTL channels : {events_OE.channel_ids}  |  t0 = {t0:.6f} s")

ttl_ch_id   = events_OE.channel_ids[1]   # skip 'Messages' at index 0
evs         = events_OE.get_events(channel_id=ttl_ch_id, segment_index=0)
times_ttl   = np.array(evs['time'],  dtype=np.float64) - t0
labels_ttl  = np.array(evs['label'], dtype=np.int16)

ch_tag = ttl_ch_id.replace(' ', '_')
np.save(os.path.join(events_dest, f"TTL_{ch_tag}_timestamps.npy"), times_ttl)
np.save(os.path.join(events_dest, f"TTL_{ch_tag}_labels.npy"),     labels_ttl)
print(f"Saved {len(times_ttl)} TTL events ('{ttl_ch_id}') → {events_dest}")

for probe_name, stream_info in probe_streams.items():
    ap_id  = stream_info['ap']
    lfp_id = stream_info['lfp']
    if ap_id is None:
        print(f'  Skipping {probe_name}: no AP stream found')
        continue
    full_raw_rec            = si.read_openephys(dataPath, block_index=0, stream_id=ap_id)
    parent_folder = os.path.basename(os.path.dirname(dataPath))
    subfolder = os.path.basename(dataPath)
    dest_path_rec           = os.path.join(dest_path, parent_folder, subfolder, probe_name)
    if not os.path.exists(dest_path_rec):
        os.makedirs(dest_path_rec, exist_ok=True)

    fs                      = full_raw_rec.get_sampling_frequency()
    num_segments            = full_raw_rec.get_num_segments()
    durations               = [full_raw_rec.get_num_samples(seg) / fs for seg in range(num_segments)]
    print(f"Durations (s): {durations}")

    valid_segments = [seg for seg in range(num_segments)
                    if (full_raw_rec.get_num_samples(seg) / fs) >= min_duration]

    if not valid_segments:
        print(f"  Skipping: no segments ≥ {min_duration} s")
        continue

    full_raw_rec            = full_raw_rec.select_segments(valid_segments)
    fs                      = full_raw_rec.get_sampling_frequency()
    groups                  = full_raw_rec.get_property('group')
    Chan_IDs                = full_raw_rec.channel_ids
    num_chan                = full_raw_rec.get_num_channels()
    chan_pos                = full_raw_rec.get_channel_locations()
    recName                 = os.path.basename(os.path.normpath(dataPath))
    mainDir                 = os.path.basename(os.path.dirname(dataPath))
    dtype                   = full_raw_rec.get_dtype()
    gain_value              = full_raw_rec.get_channel_gains()    # µV per count
    offset_value            = full_raw_rec.get_channel_offsets()  # offset in µV

    print(full_raw_rec)

    probe                   = full_raw_rec.get_probe()
    if probe is None:
        print('WARNING: no probe geometry found in recording')
    else:
        try:
            recording_f = recording_f.set_probe(probe)
        except Exception as e:
            print(f'WARNING: could not attach probe to recording_f: {e}')
    pg                      = ProbeGroup()
    pg.add_probe(probe)
    probeNameKS             = recName + '_' + probe_name + '_KS.prb'
    probePathKS             = os.path.join(dest_path_rec, probeNameKS)
    write_prb(probePathKS, pg)

    probeNameSI             = recName + '_' + probe_name + '_SI.json'
    probePathSI             = os.path.join(dest_path_rec, probeNameSI)
    write_probeinterface(probePathSI, probe)


    # Select LFP stream if present (NP1.0), otherwise derive from AP (NP2.0)
    if lfp_id is not None:
        lfp_raw_rec = si.read_openephys(dataPath, block_index=0, stream_id=lfp_id)
    else:
        lfp_raw_rec = full_raw_rec

    # -------------------------------
    # LFP Processing
    # -------------------------------
    lfp_folder    = os.path.join(dest_path_rec, 'lfp')
    os.makedirs(lfp_folder, exist_ok=True)

    shank_folders = sorted([
        f for f in os.listdir(lfp_folder)
        if os.path.isdir(os.path.join(lfp_folder, f)) and f.startswith('shank_')
    ])

    if reRunLFP or not shank_folders:
        # I. Preprocess — no scale here, matching reference approach
        
        # Validate LFP segments and drop short ones (in case LFP is separate stream)
        lfp_fs = lfp_raw_rec.get_sampling_frequency()
        lfp_num_segments = lfp_raw_rec.get_num_segments()
        lfp_durations = [lfp_raw_rec.get_num_samples(seg) / lfp_fs for seg in range(lfp_num_segments)]
        lfp_valid_segments = [seg for seg in range(lfp_num_segments) if lfp_durations[seg] >= min_duration]
        if not lfp_valid_segments:
            print(f'  Skipping LFP: no segments ? {min_duration} s')
            recordings_by_shank = {}
        else:
            lfp_raw_rec = lfp_raw_rec.select_segments(lfp_valid_segments)
        recording_shift = phase_shift(recording=lfp_raw_rec)
        LFP_filtered    = si.bandpass_filter(recording_shift, freq_min=0.1, freq_max=300)
        downsample_LFP  = si.resample(LFP_filtered, resample_rate=resampled_fs)

        # II. Split by shank — NP2: split_by('group'); NP1: single shank
        lfp_groups = lfp_raw_rec.get_property('group')
        if lfp_groups is not None and len(np.unique(lfp_groups)) > 1:
            recordings_by_shank = downsample_LFP.split_by('group')
        else:
            recordings_by_shank = {0: downsample_LFP}

        shank_ids = list(recordings_by_shank.keys())
        if len(shank_ids) == 0:
            print('  Skipping LFP: no shanks available after segment filtering')
            continue
        n_shanks  = len(shank_ids)

        # Chunk size heuristic: NP1.0 (single shank / 384ch) needs smaller chunks for phase_shift FFT
        if n_shanks > 1:
            lfp_chunk_duration = '10s'
        else:
            lfp_chunk_duration = '2s'

        # III. Plot — all shanks side by side with GWPmap
        shank_ids = sorted(recordings_by_shank.keys())
        n_shanks  = len(shank_ids)

        fig_path = os.path.join(lfp_folder, f"{recName}_{probe_name}_LFP_traces.png")
        if os.path.exists(fig_path):
            print(f"LFP figure already exists ? {fig_path} (skipping plot/save)")
        else:
            LFP_fig, axs = plt.subplots(ncols=n_shanks, figsize=(5 * n_shanks, 10))
            if n_shanks == 1:
                axs = [axs]
            for ax, shank_id in zip(axs, shank_ids):
                rec = recordings_by_shank[shank_id]
                t0 = rec.get_times(segment_index=0)[0]  # absolute start time, the resampled rec may not start at 0
                
                if len(rec.channel_ids) == 0 or rec.get_num_samples() == 0:
                    print(f"Skipping empty shank {shank_id}")
                    continue
                si.plot_traces(rec,
                            time_range=(t0 + t_range_LFP[0], t0 + t_range_LFP[1]), # use this instead of time_range bacause the re-sampled recording may not start at 0
                            cmap=GWPmap,
                            order_channel_by_depth = True, 
                            backend='matplotlib',
                            clim=(-50,50),
                            ax=ax)
                ax.set_title(f"Shank {shank_id}")            
                plt.tight_layout()
                LFP_fig.savefig(fig_path, dpi=150)
                plt.show()
                print(f"LFP figure saved ? {fig_path}")

        # IV. Save — apply µV scale per shank at save time (avoids scale+split_by issue)
        for shank_id, rec in recordings_by_shank.items():
            ch_mask      = np.isin(lfp_raw_rec.channel_ids, rec.channel_ids)
            shank_gain   = lfp_raw_rec.get_channel_gains()[ch_mask]
            shank_offset = lfp_raw_rec.get_channel_offsets()[ch_mask]
            rec_uv       = si.scale(rec, gain=shank_gain, offset=shank_offset, dtype='float32')
            shank_path   = os.path.join(lfp_folder, f"shank_{shank_id}")
            # rec_uv.save(folder=shank_path, format='binary', n_jobs=1, chunk_duration='10s', overwrite=True)                
            if os.path.isdir(shank_path) and any(os.scandir(shank_path)):
                print(f"LFP shank {shank_id} already exists ? {shank_path} (skipping save)")
            else:
                rec_uv.save(folder=shank_path, format='binary', n_jobs=1, chunk_duration=lfp_chunk_duration, overwrite=True)
                print(f"LFP shank {shank_id} saved ? {shank_path}")

    else:
        print(f"Skipping LFP: {len(shank_folders)} shank folder(s) already exist in {lfp_folder}")

    # -------------------------------
    # Spike-band preprocessing
    # -------------------------------
    recording_shift         = phase_shift(recording=full_raw_rec)
    recording_cmr           = common_reference(recording=recording_shift, operator="median")
    recording_f = bandpass_filter(
        recording=recording_cmr,
        freq_min=500,
        freq_max=6000
    )

    # -------------------------------
    # 2️⃣ Define preprocessed folder (directly on destination drive)
    # -------------------------------
    preprocesseDir = os.path.join(dest_path_rec, 'preprocessed')
    os.makedirs(preprocesseDir, exist_ok=True)

    # -------------------------------
    # 3️⃣ Save or load binary recording
    # -------------------------------
    raw_files = [f for f in os.listdir(preprocesseDir) if f.endswith('.raw')]

    binary_dtype = "int16"

    do_preprocessing = reDoPreprocessing or (len(raw_files) == 0)

    if do_preprocessing:
        print("Running preprocessing (save raw binary)...")
        recording_to_save = si.astype(recording_f, dtype=binary_dtype)
        recording_saved = recording_to_save.save(
            folder=preprocesseDir,
            n_jobs=-1,
            chunk_duration='1s',
            overwrite=True
        )
    else:
        raw_file_path = os.path.join(preprocesseDir, raw_files[0])
        print(f"Loading existing preprocessed binary from {raw_file_path}")
        recording_saved = si.read_binary(
            raw_file_path,
            sampling_frequency=fs,
            num_channels=num_chan,
            dtype=binary_dtype,
            channel_ids=Chan_IDs,
            gain_to_uV=gain_value,
            offset_to_uV=offset_value
        )
        recording_saved = recording_saved.set_probe(probe)

    # Ensure probe geometry is attached for Kilosort/Phy
    if probe is not None:
        try:
            recording_saved = recording_saved.set_probe(probe)
        except Exception as e:
            print(f'WARNING: could not attach probe to recording_saved: {e}')

    # -------------------------------
    # 4️⃣ Kilosort4 via SpikeInterface
    # -------------------------------
    kilosort_folder = os.path.join(preprocesseDir, "kilosort4")
    sorter_output_spike_times = os.path.join(kilosort_folder, "sorter_output", "spike_times.npy")

    needs_rerun_kilosort = reRunKilosort or not os.path.exists(sorter_output_spike_times)

    if needs_rerun_kilosort:
        print(f"Running Kilosort4 (reRunKilosort={reRunKilosort})... ")
        sorting = si.run_sorter(
            sorter_name="kilosort4",
            recording=recording_saved,
            folder=kilosort_folder,
            remove_existing_folder=True,
            verbose=True,
            whitening_range=32,
            dminx=32,
            nblocks=5,
            do_CAR=False,
            )

        # kilosort writes binary.json in the preprocessed folder, not inside kilosort_folder
        binary_json_path = os.path.normpath(os.path.join(preprocesseDir, "binary.json"))
        if not os.path.exists(binary_json_path):
            raise FileNotFoundError(f"Expected binary.json not found at {binary_json_path} after run_sorter")

        if not os.path.exists(sorter_output_spike_times):
            raise FileNotFoundError(f"Expected spike_times.npy not found at {sorter_output_spike_times} after Kilosort")
    else:
        print(f"Skipping Kilosort: found existing sorter output with spike_times.npy")
        sorting = si.read_sorter_folder(kilosort_folder)

    # -------------------------------
    # 5️⃣ SortingAnalyzer for automated curation
    # -------------------------------
    analyzer_folder = os.path.join(kilosort_folder, "analyzer")
    analyzer_export_folder = os.path.join(kilosort_folder, "analyzer_export")
    phy_folder      = os.path.join(kilosort_folder, "phy_export")

    os.makedirs(analyzer_folder, exist_ok=True)

    if reRunAnalyzer or not os.path.exists(os.path.join(analyzer_folder, "random_spikes.npz")):
        analyzer = si.create_sorting_analyzer(
            sorting=sorting,
            recording=recording_saved,
            format="memory",
            folder=analyzer_folder,
            overwrite=True
        )
        analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=600, seed=2205)
        analyzer.compute("waveforms", ms_before=1.3, ms_after=2.6, n_jobs=-1)
        analyzer.compute("templates", operators=["average", "median", "std"])
        analyzer.compute("template_metrics", include_multi_channel_metrics=True)
        analyzer.compute("template_metrics", metric_names=["spread", "exp_decay", "velocity_above", "velocity_below"])
        analyzer.compute("principal_components", n_components=3, mode="by_channel_global", whiten=True, n_jobs=-1)
        exts = ['noise_levels','spike_locations','spike_amplitudes','correlograms','quality_metrics',
                            'unit_locations','template_similarity']
        try:
            analyzer.compute(exts, n_jobs=-1)
        except Exception as e:
            print(f"Parallel compute failed, falling back to serial. Error: {repr(e)}")
            analyzer.compute(exts, n_jobs=1)

        si.export_to_phy(
            analyzer,
            output_folder=phy_folder,
            copy_binary=False,
            n_jobs=-1
            )

        noise_neuron_labels = sc.auto_label_units(
            sorting_analyzer=analyzer,
            repo_id="SpikeInterface/UnitRefine_noise_neural_classifier",
            trust_model=True,
            export_to_phy=True
        )
        noise_units = noise_neuron_labels[noise_neuron_labels['prediction']=='noise']
        analyzer_neural = analyzer.remove_units(noise_units.index)

        sua_mua_labels = sc.auto_label_units(
            sorting_analyzer=analyzer_neural,
            repo_id="SpikeInterface/UnitRefine_sua_mua_classifier",
            trust_model=True,
            export_to_phy=True
        )

        analyzer.save_as(folder=analyzer_export_folder, format="binary_folder")
        all_labels = pd.concat([sua_mua_labels, noise_units]).sort_index()
        all_labels.to_csv(os.path.join(analyzer_export_folder, "unit_labels.csv"))
    else:
        print(f"Analyzer already exists at {analyzer_folder}, loading existing analyzer")
        analyzer = si.load_sorting_analyzer(analyzer_folder)
        labels_path = os.path.join(analyzer_folder, "unit_labels.csv")
        if os.path.exists(labels_path):
            all_labels = pd.read_csv(labels_path, index_col=0)
        else:
            print(f"Labels file not found at {labels_path}, unable to load labels")
            all_labels = None

    print(analyzer)


Processing: I:\Neuropixels_Backup\Agata\20251219_Agata_VNS_N6_happy\2025-12-19_16-06-59
Probes found: ['ProbeA']
Streams by probe: {'ProbeA': {'ap': '0', 'lfp': '1', 'other': []}}
TTL channels : ['Messages' 'Neuropixels PXI Sync' 'Neuropixels PXI Sync']  |  t0 = 2225.160533 s
Saved 1894 TTL events ('Neuropixels PXI Sync') → X:\Agata_spikesorting\20251219_Agata_VNS_N6_happy\2025-12-19_16-06-59\events
Durations (s): [3685.697866666667]
SelectSegmentRecording: 384 channels - 30.0kHz - 1 segments - 110,570,936 samples 
                        3,685.70s (1.02 hours) - int16 dtype - 79.09 GiB
Skipping LFP: 1 shank folder(s) already exist in X:\Agata_spikesorting\20251219_Agata_VNS_N6_happy\2025-12-19_16-06-59\ProbeA\lfp
Loading existing preprocessed binary from X:\Agata_spikesorting\20251219_Agata_VNS_N6_happy\2025-12-19_16-06-59\ProbeA\preprocessed\traces_cached_seg0.raw
Skipping Kilosort: found existing sorter output with spike_times.npy


c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\spikeinterface\core\basesorting.py:316: UserWarning: Some spikes exceed the recording's duration! Removing these excess spikes with `spikeinterface.curation.remove_excess_spikes()` Might be necessary for further postprocessing.
  warnings.warn(


estimate_sparsity (no parallelization):   0%|          | 0/3686 [00:00<?, ?it/s]

c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\spikeinterface\core\sortinganalyzer.py:377: UserWarning: Your sorting has spikes with samples times greater than your recording length. These spikes have been removed.
  warnings.warn(


compute_waveforms (workers: 32 processes):   0%|          | 0/3686 [00:00<?, ?it/s]

c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\spikeinterface\postprocessing\template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\spikeinterface\postprocessing\template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\spikeinterface\postprocessing\template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\spikeinterface\postprocessing\template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
c:\Users\Andrea\.conda\envs\spikesort\lib\site-packages\sklearn\linear_model\_theil_sen.py:128: ConvergenceWarning: Maximum number of iterations

Fitting PCA:   0%|          | 0/1110 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/1110 [00:00<?, ?it/s]

noise_level (workers: 20 processes):   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
import os
import spikeinterface as si

analyzer_folder = r"C:\tmp\spike_analyzer_cache"
os.makedirs(analyzer_folder, exist_ok=True)

analyzer = si.create_sorting_analyzer(
    sorting=sorting,
    recording=recording_saved,
    folder=analyzer_folder,
    overwrite=True
)

# recompute your extensions
analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=600, seed=2205)
analyzer.compute("waveforms", ms_before=1.3, ms_after=2.6, n_jobs=-1)
analyzer.compute("templates", operators=["average", "median", "std"])
analyzer.compute("template_metrics", include_multi_channel_metrics=True)
analyzer.compute("template_metrics", metric_names=["spread", "exp_decay", "velocity_above", "velocity_below"])
analyzer.compute("principal_components", n_components=3, mode="by_channel_global", whiten=True, n_jobs=-1)
# analyzer.compute(['noise_levels','spike_locations','spike_amplitudes','correlograms','quality_metrics',
#                 'unit_locations','template_similarity'],
#                 n_jobs=-1)
# run serial first to confirm it’s stable
analyzer.compute(
    ['noise_levels','spike_locations','spike_amplitudes','correlograms','quality_metrics',
     'unit_locations','template_similarity'],
    n_jobs=1
)


In [ ]:
# ── Raw trace overview per shank ───────────────────────────────────────────────
# Requires full_raw_rec to be in scope (run main loop first, or load manually).
# Splits by shank group and plots a short window with GWPmap colouring.

t_range_raw = [60, 62]   # seconds from recording start — adjust as needed
clim_raw    = (-100, 100) # µV limits for raw data — adjust to your signal

groups_raw = full_raw_rec.get_property('group')
if groups_raw is not None and len(np.unique(groups_raw)) > 1:
    recs_by_shank_raw = full_raw_rec.split_by('group')
else:
    recs_by_shank_raw = {0: full_raw_rec}

shank_ids_raw = list(recs_by_shank_raw.keys())
n_shanks_raw  = len(shank_ids_raw)

fig_raw, axs_raw = plt.subplots(ncols=n_shanks_raw, figsize=(5 * n_shanks_raw, 10))
if n_shanks_raw == 1:
    axs_raw = [axs_raw]

for ax, shank_id in zip(axs_raw, shank_ids_raw):
    rec = recs_by_shank_raw[shank_id]
    t0  = rec.get_times(segment_index=0)[0]  # absolute start time

    if len(rec.channel_ids) == 0 or rec.get_num_samples() == 0:
        print(f"Skipping empty shank {shank_id}")
        continue

    si.plot_traces(
        rec,
        time_range=(t0 + t_range_raw[0], t0 + t_range_raw[1]),
        cmap=GWPmap,
        backend='matplotlib',
        clim=clim_raw,
        ax=ax,
    )
    ax.set_title(f"Shank {shank_id}")

fig_raw.suptitle(f"Raw trace overview — {recName}  {probes[id_num]}", fontsize=11)
plt.tight_layout()
plt.show()

### Plot with ephyviewer

In [ ]:
# plot 5 seconds starting at 10 s
# w_ts = sw.plot_traces(full_raw_rec, mode="map", time_range=(5, 6), show_channel_ids=False, order_channel_by_depth=True)
# si.plot_traces(full_raw_rec, mode="map", time_range=(5, 6), show_channel_ids=False, order_channel_by_depth=True)
sig_source = ephyviewer.SpikeInterfaceRecordingSource(recording=recording_f)
app = ephyviewer.mkQApp()
win = ephyviewer.MainViewer(debug=True, show_auto_scale=True)

view = ephyviewer.TraceViewer(source=sig_source, name='signals')
win.add_view(view)
win.show() 
app.exec()